In [ ]:
import os
import gc
import time
import pandas as pd
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# ==============================================================================
# 🎛️ PARAMETER KONFIGURASI JALUR LOKAL & FILTER 
# ==============================================================================
PATH_KATALOG_MASTER = '/Volumes/Local Disk/Code_Git/S3_code/seismic/usgs_katalog/katalog_usgs_master_2001_2025.csv'
#PATH_KATALOG_MASTER = '/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/BMKG_Earthquake_Catalog.csv'
#PATH_KATALOG_MASTER = '/Volumes/Local Disk/Code_Git/S3_code/seismic/katalog_radar_dll/HYBRID_EARTHQUAKE_CATALOG_2001_2024.csv'

PATH_CSV_FINAL = '/Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_indonesia_usgs_bmkg_katalog/katalog_radar_dll/INDONESIA_STATION_INVENTORY_FINAL.csv'
OUTPUT_WAVEFORM_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_indonesia_usgs_bmkg_katalog/output_waveform_indonesia_0109'

LOG_ERROR_MURNI_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_indonesia_usgs_bmkg_katalog/log_error/error_log_murni_0109.csv'
LOG_SUCCESS_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_indonesia_usgs_bmkg_katalog/log_success/success_log_0109.csv'
LOG_FALLBACK_GFZ_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_indonesia_usgs_bmkg_katalog/log_error/failed_iris_need_gfz_0109.csv'

# ==============================================================================
# 🎚️ PARAMETER AMBANG BATAS TARGET BARU TAHUN 2006-2007 & KUOTA 200 DATA
# ==============================================================================
START_YEAR = 2001
END_YEAR = 2005
MIN_MAGNITUDE = 3.5
MAX_MAGNITUDE = 9.5

# 📌 TARGET KUOTA BARU: Maksimal mengambil 200 baris rekaman komponen per Event ID
MAX_DATA_PER_EVENT = 200 
MAX_WORKERS = 20 

ACADEMIC_USER_AGENT = (
    "ResearchProject: Doctoral Dissertation in AI and Edge Computing; "
    "Researcher: Very Kurnia Bakti (Indonesia); "
    "ID: Scopus:57209452703"
)

clients = {
    "IRIS": Client("IRIS", timeout=60, user_agent=ACADEMIC_USER_AGENT),
    "GFZ": Client("IRIS", timeout=60, user_agent=ACADEMIC_USER_AGENT)
}

COMPLETED_TASKS_SET = set()

def download_single_waveform(task_data):
    eid, time_str, eq_lat, eq_lon, net, sta, server = task_data
    task_key = f"{eid}_{net}_{sta}"
    
    if task_key in COMPLETED_TASKS_SET:
        return "SKIPPED", None
        
    try:
        t_event = UTCDateTime(time_str)
        year_folder = str(t_event.year)
        event_dir = os.path.join(OUTPUT_WAVEFORM_DIR, year_folder, str(eid))
        filename = f"{net}_{sta}_{eid}.mseed"
        file_path = os.path.join(event_dir, filename)
        
        if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
            return "SKIPPED", None
            
        cl = clients.get(server)
        if not cl:
            return "ERROR_SERVER", (eid, time_str, net, sta, server, "Server Config Error")
            
        start_time = t_event - 15
        end_time = t_event + 120
        
        # Ambil data gelombang mentah dan tampung sementara di RAM
        st = cl.get_waveforms(network=net, station=sta, location="*", channel="BH*,HH*,EH*",
                              starttime=start_time, endtime=end_time)
        
        # ==============================================================================
        # 🛡️ SENSOR PENGAMAN: VALIDASI KELENGKAPAN TRI-KOMPONEN (N, Z, E / 1, 2, Z)
        # ==============================================================================
        # Ambil huruf terakhir dari nama channel (e.g., 'BHZ' -> 'Z', 'BHN' -> 'N')
        found_channels = set([tr.stats.channel[-1].upper() for tr in st])
        
        # Definisikan subset komponen lengkap standar seismologi global
        has_standard_nze = {'Z', 'N', 'E'}.issubset(found_channels)
        has_orthogonal_z12 = {'Z', '1', '2'}.issubset(found_channels)
        
        # Jika fasa komponen tidak lengkap, gugurkan penulisan file fisik ke SSD Mac
        if not (has_standard_nze or has_orthogonal_z12):
            channels_logged = ",".join(list(found_channels))
            # Hancurkan stream dari RAM agar terhindar dari memory leak
            del st
            return "FAILED_INCOMPLETE_CHANNELS", (eid, sta, f"Missing Components (Found: {channels_logged})")
        # ==============================================================================
        
        # Jika lolos sensor kelengkapan, eksekusi pembuatan folder dan simpan biner .mseed
        os.makedirs(event_dir, exist_ok=True)
        st.write(file_path, format="MSEED")
        del st
        
        time.sleep(0.05) 
        return "SUCCESS", task_key
            
    except Exception as e:
        err_name = type(e).__name__
        if err_name in ["HTTPError", "FDSNTimeoutException", "ConnectionError", "TimeoutError"]:
            return "NEED_GFZ", (eid, time_str, net, sta, server, err_name)
        else:
            return f"FAILED_{err_name}", (eid, time_str, net, sta, server, err_name)

def build_success_index_from_storage():
    global COMPLETED_TASKS_SET
    print("🔍 Menginisialisasi Indeks Turbo Resume...")
    os.makedirs(os.path.dirname(LOG_SUCCESS_PATH), exist_ok=True)
    
    if os.path.exists(LOG_SUCCESS_PATH):
        try:
            df_succ = pd.read_csv(LOG_SUCCESS_PATH)
            COMPLETED_TASKS_SET = set(df_succ['Task_Key'].astype(str).tolist())
            print(f"✅ Berhasil memuat {len(COMPLETED_TASKS_SET):,} file sukses ke RAM Mac!")
            return
        except Exception:
            pass

    print("📂 Menyisir folder Local Disk untuk mendata file sukses...")
    scanned_keys = []
    if os.path.exists(OUTPUT_WAVEFORM_DIR):
        for root, _, files in os.walk(OUTPUT_WAVEFORM_DIR):
            for file in files:
                if file.endswith('.mseed'):
                    parts = file.replace('.mseed', '').split('_')
                    if len(parts) >= 3:
                        scanned_keys.append(f"{parts[2]}_{parts[0]}_{parts[1]}")
                        
    COMPLETED_TASKS_SET = set(scanned_keys)
    if scanned_keys:
        pd.DataFrame({"Task_Key": scanned_keys}).to_csv(LOG_SUCCESS_PATH, index=False)
    print(f"✅ Sinkronisasi Selesai! {len(COMPLETED_TASKS_SET):,} file terdata di RAM.")

def run_closest_station_pipeline():
    print(f"🛡️  Starting Fixed-Count Seismology Downloader Pipeline (Max {MAX_DATA_PER_EVENT} Data Per Event)...")
    os.makedirs(os.path.dirname(LOG_ERROR_MURNI_PATH), exist_ok=True)
    os.makedirs(os.path.dirname(LOG_FALLBACK_GFZ_PATH), exist_ok=True)
    
    build_success_index_from_storage()
    
    if not os.path.exists(PATH_CSV_FINAL) or not os.path.exists(PATH_KATALOG_MASTER):
        print("❌ Berkas peta navigasi inventory final atau katalog master tidak ditemukan!")
        return
        
    print("⏳ Loading master navigation footprint & earthquake catalog...")
    df_inventory = pd.read_csv(PATH_CSV_FINAL)
    df_master = pd.read_csv(PATH_KATALOG_MASTER)
    
    col_master_id = next((c for c in df_master.columns if 'id' in c.lower()), 'id')
    col_master_lat = next((c for c in df_master.columns if 'lat' in c.lower()), 'latitude')
    col_master_lon = next((c for c in df_master.columns if 'lon' in c.lower()), 'longitude')
    
    df_master_clean = df_master[[col_master_id, col_master_lat, col_master_lon]].copy()
    df_master_clean.columns = ['Event_ID', 'Eq_Latitude', 'Eq_Longitude']
    
    df_inventory['Event_ID'] = df_inventory['Event_ID'].astype(str)
    df_master_clean['Event_ID'] = df_master_clean['Event_ID'].astype(str)
    
    df_merged = pd.merge(df_inventory, df_master_clean, on='Event_ID', how='inner')
    
    if len(df_merged) == 0:
        print("❌ Gagal mencocokkan data! Tidak ada Event_ID yang selaras antara kedua file.")
        return

    col_time = 'Time_UTC'
    col_mag = 'Mag'
    
    df_merged[col_time] = pd.to_datetime(df_merged[col_time], errors='coerce')
    df_filtered = df_merged[
        (df_merged[col_time] >= f"{START_YEAR}-01-01") & 
        (df_merged[col_time] <= f"{END_YEAR}-12-31 23:59:59") &
        (df_merged[col_mag] >= MIN_MAGNITUDE) & 
        (df_merged[col_mag] <= MAX_MAGNITUDE)
    ].copy()
    
    if len(df_filtered) == 0:
        print("⚠️ Tidak ada data yang cocok dengan kriteria rentang target di memori.")
        return

    # Penomoran baris rekaman kumulatif alami per Event_ID (Stasiun dibebaskan)
    df_filtered['Data_Rank'] = df_filtered.groupby('Event_ID').cumcount() + 1
    
    # Saring kuota tugas secara konsisten maksimal 200 data per event gempa
    df_final_tasks = df_filtered[df_filtered['Data_Rank'] <= MAX_DATA_PER_EVENT].copy()
    
    event_counts = df_final_tasks.groupby('Event_ID').size()
    valid_eids = event_counts[event_counts >= 2].index
    df_final_tasks = df_final_tasks[df_final_tasks['Event_ID'].isin(valid_eids)]
    
    tasks = list(df_final_tasks[['Event_ID', 'Time_UTC', 'Eq_Latitude', 'Eq_Longitude', 'Net', 'Station', 'Server']].itertuples(index=False, name=None))
    total_tasks = len(tasks)
    unique_events = len(valid_eids)
    
    del df_master, df_inventory, df_master_clean, df_merged, df_filtered, df_final_tasks
    gc.collect()
    
    print(f"📊 Filter Mengunci: {unique_events:,} Kejadian Gempa Bumi Unik.")
    print(f"📡 Total Antrean Unduhan Sinyal (Maksimal {MAX_DATA_PER_EVENT} Data per Event): {total_tasks:,} Berkas Tugas.")
    
    stats = {"SUCCESS": 0, "SKIPPED": 0, "FAILED": 0, "FALLBACK": 0}
    error_logs, fallback_logs, new_success_keys = [], [], []
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        pbar = tqdm(total=total_tasks, unit="file", desc="Harvesting Waveforms")
        
        for status, detail in executor.map(download_single_waveform, tasks):
            if status == "SUCCESS":
                stats["SUCCESS"] += 1
                if detail: new_success_keys.append({"Task_Key": detail})
            elif status == "SKIPPED":
                stats["SKIPPED"] += 1
            elif status == "NEED_GFZ":
                stats["FALLBACK"] += 1
                if detail: fallback_logs.append({"Event_ID": detail[0], "Time_UTC": detail[1], "Net": detail[2], "Station": detail[3], "Server": "GFZ", "Reason": detail[5]})
            else:
                stats["FAILED"] += 1
                if detail: error_logs.append({"Event_ID": detail[0], "Station": detail[1], "Error_Reason": detail[2]})
                
            pbar.update(1)
            pbar.set_postfix({"New": stats["SUCCESS"], "Skip": stats["SKIPPED"], "To_GFZ": stats["FALLBACK"], "NoData": stats["FAILED"]})
            
            total_processed = stats["SUCCESS"] + stats["SKIPPED"] + stats["FAILED"] + stats["FALLBACK"]
            if total_processed % 400 == 0:
                gc.collect()
                if new_success_keys and len(new_success_keys) >= 1000:
                    pd.DataFrame(new_success_keys).to_csv(LOG_SUCCESS_PATH, mode='a', header=not os.path.exists(LOG_SUCCESS_PATH), index=False)
                    new_success_keys.clear()
                if fallback_logs and len(fallback_logs) >= 500:
                    pd.DataFrame(fallback_logs).to_csv(LOG_FALLBACK_GFZ_PATH, mode='a', header=not os.path.exists(LOG_FALLBACK_GFZ_PATH), index=False)
                    fallback_logs.clear()

    pbar.close()
    
    if new_success_keys: pd.DataFrame(new_success_keys).to_csv(LOG_SUCCESS_PATH, mode='a', header=not os.path.exists(LOG_SUCCESS_PATH), index=False)
    if fallback_logs: pd.DataFrame(fallback_logs).to_csv(LOG_FALLBACK_GFZ_PATH, mode='a', header=not os.path.exists(LOG_FALLBACK_GFZ_PATH), index=False)
    if error_logs: pd.DataFrame(error_logs).to_csv(LOG_ERROR_MURNI_PATH, index=False)
        
    print("\n" + "="*50 + "\n🏁 PIPELINE PENGUNDUHAN KOREKSI KELOMPOK SELESAI\n" + "="*50)
    print(f"✅ Berkas Baru Sukses Terjemput      : {stats['SUCCESS']:,} file")
    print(f"🔄 Berkas Lama Aman Terlewati        : {stats['SKIPPED']:,} file")
    print(f"⚠️ Masalah Jaringan (Dialihkan GFZ) : {stats['FALLBACK']:,} file")
    print(f"❌ Berkas Gagal (Absen/Tidak Lengkap): {stats['FAILED']:,} file")
    print("="*50)

if __name__ == "__main__":
    run_closest_station_pipeline()

/opt/homebrew/Caskroom/miniforge/base/envs/waveform/lib/python3.10/site-packages/obspy/clients/fdsn/client.py:251: ObsPyDeprecationWarning: IRIS is now EarthScope, please consider changing the FDSN client short URL to 'EARTHSCOPE'.
  warnings.warn(msg, ObsPyDeprecationWarning)


🛡️  Starting Fixed-Count Seismology Downloader Pipeline (Max 200 Data Per Event)...
🔍 Menginisialisasi Indeks Turbo Resume...
✅ Berhasil memuat 6,952 file sukses ke RAM Mac!
⏳ Loading master navigation footprint & earthquake catalog...
📊 Filter Mengunci: 16,544 Kejadian Gempa Bumi Unik.
📡 Total Antrean Unduhan Sinyal (Maksimal 200 Data per Event): 34,363 Berkas Tugas.
